In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-08 02:12:12.770407: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-08 02:12:13.627408: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 1,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 2,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-08 02:12:18,934 [DEBUG] [Rain] Rain is initialized
2023-07-08 02:12:18,937 [DEBUG] [Provisioner] Creating coordinator
2023-07-08 02:12:18,938 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-08 02:12:18,939 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-08 02:12:18,939 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-08 02:12:18,941 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-08 02:12:18,942 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-08 02:12:18,942 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
# model = rain.train(X_train, y_train, strategy='async')

In [9]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-08 02:12:18,970 [INFO] [Provisioner] provisioner is serving
2023-07-08 02:12:18,971 [DEBUG] [Provisioner] Starting coordinator
2023-07-08 02:12:18,973 [INFO] [Coordinator] coordinator is serving
2023-07-08 02:12:18,974 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-08 02:12:18,980 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-08 02:12:18,982 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-08 02:12:18,984 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-08 02:12:18,986 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-08 02:12:18,987 [DEBUG] [LocalProvisioner] Creating 1 workers
2023-07-08 02:12:18,988 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-08 02:12:18,990 [INFO] [Worker_50151] Worker is running 

2023-07-08 02:12:19,037 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-08 02:12:19,039 [DEBUG] [Provisioner] Received '' from the coordinator to send status
2023-07-08 02:12:19,039 [DEBUG] [Provisioner] Workers
IPs : ['127.0.0.1'], ports: [50151], statuses: [1], IDs : [1]
2023-07-08 02:12:19,040 [DEBUG] [DividerAmbassador] divider received: information from coordinator
2023-07-08 02:12:19,041 [DEBUG] [DeepLearning] Starting iteration 1/2
2023-07-08 02:12:19,067 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-08 02:12:26,228 [DEBUG] [DividerAmbassador] divider receive: File downloaded successfully from  worker after sending X_train
2023-07-08 02:12:26,289 [DEBUG] [DividerAmbassador] divider receive: File downloaded successfully from worker 1 after sending y_train
2023-07-08 02:12:26,290 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/1.pkl to worker1
2023-07-08 02:12:26,347 [DEBUG] [DividerAmbassador] divider received: File downloaded suc

Epoch 1/5
469/469 [==============================] - 3s 5ms/step - loss: 0.4273 - accuracy: 0.8687
Epoch 2/5
469/469 [==============================] - 2s 4ms/step - loss: 0.1953 - accuracy: 0.9414
Epoch 3/5
469/469 [==============================] - 1s 3ms/step - loss: 0.1536 - accuracy: 0.9528
Epoch 4/5
469/469 [==============================] - 1s 3ms/step - loss: 0.1286 - accuracy: 0.9611
Epoch 5/5
469/469 [==============================] - 1s 3ms/step - loss: 0.1130 - accuracy: 0.9656


2023-07-08 02:12:35,600 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-08 02:12:35,600 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-08 02:12:35,652 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-08 02:12:35,673 [DEBUG] [DeepLearning] Iteration 1/2 complete.
2023-07-08 02:12:35,673 [DEBUG] [DeepLearning] Starting iteration 2/2
2023-07-08 02:12:35,692 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-08 02:12:35,693 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 1
2023-07-08 02:12:35,694 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/1.pkl to worker1
2023-07-08 02:12:35,753 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-08 02:12:35,754 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker1
2023-07-08 02:12:35,755 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2


Epoch 1/5
469/469 [==============================] - 2s 2ms/step - loss: 0.1055 - accuracy: 0.9677
Epoch 2/5
469/469 [==============================] - 1s 2ms/step - loss: 0.0945 - accuracy: 0.9710
Epoch 3/5
469/469 [==============================] - 1s 3ms/step - loss: 0.0902 - accuracy: 0.9718
Epoch 4/5
469/469 [==============================] - 1s 3ms/step - loss: 0.0839 - accuracy: 0.9728
Epoch 5/5
469/469 [==============================] - 1s 3ms/step - loss: 0.0772 - accuracy: 0.9752


2023-07-08 02:12:42,498 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-08 02:12:42,499 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_2_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_2_trained.pkl from worker1


sending data to divider


2023-07-08 02:12:42,559 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_2_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_2_trained.pkl from worker1 successfully
2023-07-08 02:12:42,581 [DEBUG] [DeepLearning] Iteration 2/2 complete.
DEBUG:DeepLearning:Iteration 2/2 complete.
2023-07-08 02:12:42,582 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-08 02:12:42,583 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0695 - accuracy: 0.9797

Test accuracy: 98.0%


2023-07-08 02:17:41,475 [DEBUG] [Provisioner] Received 'NumOfWorkers: 2
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 2
' from the coordinator to define the number of workers
